#### 1. Set up the environment

In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve().parent

INSTALL_SCRIPT = PROJECT_DIR / "setup" / "install_requirements.py"
REQUIREMENTS_FILE = PROJECT_DIR / "setup" / "requirements.yml"

assert INSTALL_SCRIPT.exists(), INSTALL_SCRIPT
assert REQUIREMENTS_FILE.exists(), REQUIREMENTS_FILE

print("Project dir:", PROJECT_DIR)
print("Install script:", INSTALL_SCRIPT)
print("Requirements:", REQUIREMENTS_FILE)

!python "{INSTALL_SCRIPT}" --requirements "{REQUIREMENTS_FILE}" --project-dir "{PROJECT_DIR}"

Project dir: /teamspace/studios/this_studio/emotion-detection
Install script: /teamspace/studios/this_studio/emotion-detection/setup/install_requirements.py
Requirements: /teamspace/studios/this_studio/emotion-detection/setup/requirements.yml



Project directory: /teamspace/studios/this_studio/emotion-detection
Git repos directory: /teamspace/studios/this_studio/emotion-detection/code

Python environment:
  executable: /teamspace/studios/this_studio/.conda/bin/python
  version:    3.11.15
yaml import failed: ModuleNotFoundError: No module named 'yaml'
Installing PyYAML so requirements.yml can be read...

Running:
/teamspace/studios/this_studio/.conda/bin/python -m pip install --upgrade pyyaml
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 103.8 MB/s  0:00:00

No git section found. Skipping repo clone.

Initial torch status:
  installed:      false
  error:          ModuleNotFoundError: No module named 'torch'

Torch does not satisfy requirements. Reinstalling torch stack...

Installing torch stack:
  - torch
  - torchvision
  - torchaudio

Running:
/teamspace/studios/this_studio/.conda/bin/python -m pip uninstall -y torch torchvision torchaudio

Running:
/teamspace/studios/this_studio/.conda/bin/python -m pip ins

#### 2. Preprocess data and train models

In [ ]:
# Imports modules
import importlib
import torch
import os
import utils
from huggingface_hub import login
import importlib
import utils
importlib.reload(utils)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "0"
read_data, prep, dls, mods, trainers = utils.import_modules(
    ["read_data", "preprocessing", "dataloaders", "models", "trainers"]
)

# Set HF Token
hf_token = ""
login(token=hf_token)

# Enable TRAINING mode to train the models (if disabled, the trained model checkpoints will be used)
TRAINING = True

# Enable hyperparameter optimization (if disabled, the default values in train_optuna.yml will be used)
HP_OPT = True

# Enable DEBUG mode to run the pipeline on a small data subset
DEBUG = False

# Set device, seed, and batch_size
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
seed = 42
batch_size = 128

# Read data, metadata, and configuration files
(
    data,
    labels,
    label2id,
    id2label,
    trial_info,
    channels,
    sf,
    configs,
    paths,
) = read_data.read()

# Main preprocessing and training loop
models = list(configs.keys())
models = ['cbramod']
metrics = {}
ypred_test = {}
for model in models:
    # Set random seed to ensure reproducibility
    print(f"\nHandling model '{model}' {'*' * 60}")
    utils.set_seeds(seed)

    # Run model-specific preprocessing pipeline
    print(f"Preprocessing {'*' * 68}")
    data, data_spec, labels, channels = prep.preprocess(
        data=data,
        model_config=configs[model]["preprocessing"],
        sf=sf,
        DEBUG=DEBUG,
        labels=labels,
        channels=channels,
    )

    # Create dataloaders
    print(f"Creating trainin/validation/test dataloaders {'*' * 38}")
    dataloaders = dls.make_dataloaders(data, labels, batch_size, seed)

    # Train models (after hp optimization)
    if TRAINING:
        print(f"Training models{'*' * 60}")
        # print(f"Running hyperparameter optimization{'*' * 38}")
        (
            best_model,
            metrics[model],
            best_hyperparams,
        ) = trainers.optimize_model(
            model=model,
            dataloaders=dataloaders,
            channels=channels,
            data_spec=data_spec,
            n_classes=len(label2id),
            HP_OPT=HP_OPT,
            device=device,
        )
    else:
        print(f"Retrieving best trained model{'*' * 38}")
        (
            best_model,
            metrics[model],
            best_hyperparams,
        ) = trainers.retrieve_best_model(
            model=model,
            channels=channels,
            data_spec=data_spec,
            n_classes=len(label2id),
            device=device,
        )

    # Evaluate best model on test set
    print(f"Evaluating best model on test set{'*' * 38}")
    ypred_test[model] = trainers.evaluate_model(
        model=best_model,
        dataloader=dataloaders["test"],
        device=device,
        dummy_y=True,
    )["ypred"]

    # Save best model predictions
    utils.save_preds(model, ypred_test[model], id2label)

    print(metrics[model])
    print(best_hyperparams)
    print()

# Print training summary and create summary plot
summary_df, plot_path = utils.summarize_model_metrics(
    metrics=metrics,
    dataloaders=dataloaders,
    n_classes=len(label2id),
)

/teamspace/studios/this_studio/emotion-detection
/teamspace/studios/this_studio/emotion-detection
Imported latest version of 'read_data' module
Imported latest version of 'preprocessing' module
Imported latest version of 'dataloaders' module


/teamspace/studios/this_studio/.conda/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Imported latest version of 'models' module
Imported latest version of 'trainers' module

Handling model 'cbramod' ************************************************************
Preprocessing ********************************************************************
Filtering...
Extracting largest window...
Normalizing signals...
Applying remapping normalization...


[I 2026-05-27 09:47:44,602] A new study created in memory with name: no-name-dd112762-79a2-4337-bea2-6a6acd6840aa


Creating trainin/validation/test dataloaders **************************************
Training models************************************************************
Weights already exist: /teamspace/studios/this_studio/emotion-detection/code/CBraMod/pretrained_weights/pretrained_weights.pth


...

Weights already exist: /teamspace/studios/this_studio/emotion-detection/code/CBraMod/pretrained_weights/pretrained_weights.pth


[I 2026-05-27 09:54:46,847] Trial 19 finished with value: 0.25757575757575757 and parameters: {'optimizer.lr': 6.400734941906068e-06, 'optimizer.weight_decay': 2.1169767992212743e-05, 'scheduler.patience': 2}. Best is trial 2 with value: 0.2727272727272727.


Weights already exist: /teamspace/studios/this_studio/emotion-detection/code/CBraMod/pretrained_weights/pretrained_weights.pth
Training model with best hyperparameters set...


Training:   0%|          | 0/50 [00:00<?, ?it/s]

Evaluating best model on test set**************************************
{'train': {'loss': 1.6648706524460404, 'accuracy': 0.6333333333333333, 'balanced_accuracy': 0.6333333333333333, 'ypred': array([5, 2, 3, 3, 0, 0, 2, 4, 0, 1, 1, 0, 4, 4, 0, 3, 5, 4, 2, 4, 0, 0,
       0, 3, 5, 5, 0, 0, 5, 3, 5, 3, 1, 0, 3, 3, 3, 5, 2, 5, 1, 0, 4, 2,
       3, 5, 0, 2, 0, 5, 3, 2, 1, 0, 1, 1, 1, 0, 4, 5, 5, 3, 0, 1, 5, 5,
       3, 3, 3, 0, 1, 5, 0, 4, 5, 5, 0, 4, 5, 1, 0, 3, 1, 5, 3, 2, 0, 0,
       5, 5, 3, 0, 1, 1, 3, 0, 0, 0, 0, 1, 0, 3, 0, 3, 5, 1, 0, 3, 4, 1,
       0, 2, 3, 5, 5, 0, 5, 3, 0, 3, 4, 1, 0, 3, 5, 2, 3, 1, 5, 0, 4, 1,
       2, 3, 2, 0, 1, 3, 0, 0, 0, 5, 4, 3, 2, 0, 3, 3, 0, 0, 4, 1, 0, 5,
       4, 4, 2, 3, 0, 4, 3, 5, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 3, 2, 4, 2,
       5, 5, 2, 0, 1, 3, 1, 5, 5, 3, 4, 1, 3, 1, 3, 3, 5, 2, 0, 2, 4, 1,
       2, 1, 0, 0, 3, 1, 3, 3, 5, 2, 5, 3, 0, 0, 0, 4, 5, 3, 5, 4, 1, 0,
       2, 5, 4, 3, 2, 4, 1, 0, 1, 0, 1, 2, 3, 5, 0, 0, 5, 1, 3, 5, 0, 0,
    